# Gene Filtering for Validation Cohort (GSE254569)

This notebook applies gene filtering (Protein Coding + Min Expression + HVG/DEG) to the validation cohort pseudobulk data.

**Input**: `data/validation_cohort/GSE254569/pb_ct`  
**Output**: `data/validation_cohort/GSE254569/pb_ct_filtered`

In [1]:
import sys
import os
import scanpy as sc
import pandas as pd

# Add sources directory to sys.path
sys.path.append(os.path.abspath('../../sources'))

import gene_filtering

# Parameters
root_data_path = "/data/home/swkim0523/research/_SCZ_mic_ast_fin_0511/data"
val_data_path = f"{root_data_path}/validation_cohort/GSE254569"
gtf_path = f"{root_data_path}/Homo_sapiens.GRCh38.114.gtf.gz"
input_dir = f"{val_data_path}/pb_ct"
output_dir = f"{val_data_path}/pb_ct_filtered"

os.makedirs(output_dir, exist_ok=True)

cell_types = ['Astrocytes', 'Microglia']

/data/home/swkim0523/.local/lib/python3.9/site-packages/setuptools_scm/version.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import iter_entry_points


In [2]:
# Load protein coding genes from GTF
pc_ids, pc_names = gene_filtering.get_protein_coding_genes_from_gtf(gtf_path)

Protein-coding IDs: 20,120, names: 19,450


In [3]:
for ct in cell_types:
    print(f"\nProcessing Cell Type: {ct}")
    
    # 1. Load joint pseudobulk data
    pb_path = f"{input_dir}/pb_{ct}_avg.h5ad"
    if not os.path.exists(pb_path):
        print(f"File not found: {pb_path}. Skipping.")
        continue
        
    adata = sc.read_h5ad(pb_path)
    
    # 2. Split by Classification
    # In GSE254569, classification is 'Control' and 'Schizophrenia'
    adata_ctrl = adata[adata.obs['Classification'] == 'Control'].copy()
    adata_scz = adata[adata.obs['Classification'] == 'Schizophrenia'].copy()
    
    print(f"Control donors: {adata_ctrl.n_obs}, SCZ donors: {adata_scz.n_obs}")
    
    # 3. Apply Filtering logic from gene_filtering module
    # This applies Protein Coding, Min Expression, and HVG/DEG filters
    adata_ctrl_filtered, adata_scz_filtered = gene_filtering.get_filtered_pseudobulk(
        adata_ctrl, 
        adata_scz, 
        pc_names
    )
    
    # 4. Save results
    out_path_ctrl = f"{output_dir}/pb_{ct}_control_filtered"
    out_path_scz = f"{output_dir}/pb_{ct}_scz_filtered"
    
    # Save as CSV and H5AD (using gene_filtering helper)
    gene_filtering.save_df_h5ad(adata_ctrl_filtered, out_path_ctrl)
    gene_filtering.save_df_h5ad(adata_scz_filtered, out_path_scz)
    
    print(f"Saved filtered data for {ct} to {output_dir}")


Processing Cell Type: Astrocytes
Control donors: 33, SCZ donors: 36
- Applying Protein Coding Gene Filter...
# Genes after Protein Coding Gene Filter:16130, 16130
- Applying Minimum Expression Filter...
# Genes Min Expression CTRL:15302, SCZ:15586, Merged:15626
# Genes after Minimum Expression Filter:15626, 15626
- Applying HVG/DEG Filter...
# Genes HVG/DEG CTRL_HVG:11719, SCZ_HVG:11719, Merged_HVG:12074
# Genes DEG:2448
-- # final filtered genes:12318 --
Saved filtered data for Astrocytes to /data/home/swkim0523/research/_SCZ_mic_ast_fin_0511/data/validation_cohort/GSE254569/pb_ct_filtered

Processing Cell Type: Microglia
Control donors: 33, SCZ donors: 36
- Applying Protein Coding Gene Filter...
# Genes after Protein Coding Gene Filter:16130, 16130
- Applying Minimum Expression Filter...
# Genes Min Expression CTRL:14151, SCZ:13921, Merged:14258
# Genes after Minimum Expression Filter:14258, 14258
- Applying HVG/DEG Filter...
# Genes HVG/DEG CTRL_HVG:10693, SCZ_HVG:10693, Merged_HVG